# 数值与数学运算

学习目标：能按精度需求选择 Number 或 BigInt，正确解析、舍入和显示数值。

前置知识：数值类型、算术运算、布尔比较与显式类型转换。

适用版本：ECMAScript 2025（ECMA-262 第 16 版）、Node.js 24.11.0；.mjs 文件按 ES 模块运行并采用严格模式。console 是宿主输出 API。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。

配套脚本：位于 scripts/04-numbers-and-math/。

1. [main.mjs](scripts/04-numbers-and-math/main.mjs)：按正文顺序运行全部正常示例。
2. [mixed-numeric-types.mjs](scripts/04-numbers-and-math/mixed-numeric-types.mjs)：BigInt 与 Number 不能直接相加；先决定所需的数值类型。
3. [bigint-fraction.mjs](scripts/04-numbers-and-math/bigint-fraction.mjs)：非整数 Number 不能直接转换为 BigInt。
4. [bigint-zero-division.mjs](scripts/04-numbers-and-math/bigint-zero-division.mjs)：BigInt 除以零不会产生 Infinity。

Step 1：从项目根目录进入本章工作目录。

```bash
cd content/编程语言/javascript
```

Step 2：运行全部正常示例，按各片段中的输出注释核对。

```bash
node scripts/04-numbers-and-math/main.mjs
```

下文正常片段依次对应 main.mjs 中的代码；每段给出自身输入与定义。错误文件仅在相应小节单独运行。

## 1 Number、字面量与进制

Number 统一表示常见整数与小数，采用 IEEE 754 双精度二进制格式，不会因写成整数就获得任意精度。Number() 调用转换为原始数值；new Number() 创建包装对象，本章不使用包装对象。

0b、0o、0x 分别表示二、八、十六进制整数字面量；不带前缀的一般写法为十进制。e 表示十进制指数，数字分隔符 _ 只能放在允许的数字之间，便于阅读，并不改变值。BigInt 整数字面量后加 n，不能带小数点或指数部分。

toString(radix) 按进制输出字符串，radix 是 2–36 的整数；它不改变原来的数值。

```javascript
console.log(0b1010, 0o12, 0xA, 10);
console.log(1_024, 1.25e3, 1.25e-2);
console.log((255).toString(16), (10).toString(2));
console.log(0xFFn, 1_000n, Number("25"));
// 输出依次为：
// 10 10 10 10
// 1024 1250 0.0125
// ff 1010
// 255n 1000n 25
```

## 2 安全整数与浮点误差

安全整数范围是 -(2 ** 53 - 1) 到 2 ** 53 - 1；这里 ** 表示乘方。范围内每个整数都能被唯一表示。超出范围仍可能表示某些整数，但相邻整数可能落到同一个 Number，Number.isInteger() 也不能证明输入未丢精度。

0.1 等十进制小数不能用有限二进制小数精确表示，运算会舍入。Number.EPSILON 是 1 与下一个较大可表示数的间距，不是适合任何数量级的万能误差阈值。应按业务的单位、数值尺度和可接受误差设容差；需要精确计数时可用整数单位并核对安全范围。下面金额以“分”作为本例输入约定，展示时才换为元。

```javascript
const largest = Number.MAX_SAFE_INTEGER;
console.log(largest, Number.isSafeInteger(largest), Number.isSafeInteger(largest + 1));
console.log(largest + 1 === largest + 2, Number.isInteger(largest + 1));
const computed = 0.1 + 0.2;
console.log(computed, computed === 0.3);
const tolerance = 1e-12; // 本例在 1 附近做演示，不推广到任意业务输入
console.log(Math.abs(computed - 0.3) < tolerance, Number.EPSILON);
const totalCents = 10 + 20;
console.log(totalCents, (totalCents / 100).toFixed(2));
// 输出依次为：
// 9007199254740991 true false
// true true
// 0.30000000000000004 false
// true 2.220446049250313e-16
// 30 0.30
```

## 3 Infinity、NaN 与输入检查

Number 运算溢出或非零数除以零可能产生 Infinity；0 / 0 等无法得到数值结果的运算产生 NaN。它们仍属于 number。

Number.isFinite() 只接受有限的 Number，不做转换；Number.isNaN() 只判断输入是否为 NaN。全局 isFinite()、isNaN() 会先进行数值转换，容易把空白或 null 当作 0。应先按需求解析，再检查是否有限，以及范围、整数条件等业务限制。

```javascript
console.log(1 / 0, -1 / 0, 0 / 0, Number.MAX_VALUE * 2);
console.log(Number.isFinite(12), Number.isFinite("12"), isFinite("12"));
console.log(Number.isNaN("oops"), isNaN("oops"), Number.isNaN(Number("oops")));
const seats = Number("12");
console.log(Number.isSafeInteger(seats) && seats >= 0);
// 输出依次为：
// Infinity -Infinity NaN Infinity
// true false true
// false true true
// true
```

## 4 整段转换与前缀解析

Number() 要求整段字符串符合数值转换语法。Number.parseInt() 按给定进制读取整数前缀，Number.parseFloat() 读取十进制浮点前缀，后续不合语法的字符会被忽略；没有有效前缀时得到 NaN。解析成功不等于整个输入合法。

parseInt 的 radix 是 2–36 的进制整数，建议显式指定；省略时默认按十进制，但 0x 前缀会触发十六进制，0b 不会自动按二进制解析。数字分隔符属于源码字面量语法，不属于这些字符串解析格式。不要用 parseInt() 来截断已经是 Number 的小数，使用 Math.trunc()。

```javascript
console.log(Number("12px"), Number.parseInt("12px", 10), Number.parseFloat("12.5px"));
console.log(Number.parseInt("ff", 16), Number.parseInt("101", 2));
console.log(Number.parseInt("0b10"), Number("0b10"));
console.log(Number("1_000"), Number.parseInt("1_000", 10));
console.log(Number.parseInt("none", 10), Math.trunc(12.9));
// 输出依次为：
// NaN 12 12.5
// 255 5
// 0 2
// NaN 1
// NaN 12
```

## 5 Math 的计算与舍入

Math 是提供常量和静态方法的对象，不用 new 创建。下面 x 表示有限 Number；三角函数的角度以弧度计，Math.PI 是圆周率近似值。

| 完整名称 | 中文名称／含义 |
| --- | --- |
| Math.abs(x) | 绝对值 |
| Math.sqrt(x) | 平方根 |
| Math.hypot(x, y) | 平方和的平方根，x、y 为直角边长度 |
| Math.min(x, y) | 较小值 |
| Math.max(x, y) | 较大值 |
| Math.floor(x) | 向负无穷取整 |
| Math.ceil(x) | 向正无穷取整 |
| Math.trunc(x) | 去掉小数部分，向零取整 |
| Math.round(x) | 最近整数，等距时选择较靠近正无穷的值 |

负数上的舍入尤其容易混淆。部分数学函数属于实现近似运算，不能要求不同引擎的所有末位都一致。Math 方法以 Number 运算，不接受 BigInt。

```javascript
console.log(Math.abs(-6), Math.sqrt(81), Math.hypot(3, 4));
console.log(Math.min(6, 2), Math.max(6, 2), Math.sin(0));
console.log(Math.floor(-1.5), Math.ceil(-1.5), Math.trunc(-1.5), Math.round(-1.5));
console.log(Object.is(Math.round(-0.1), -0));
console.log(Math.min(), Math.max());
// 输出依次为：
// 6 9 5
// 2 6 0
// -2 -1 -1 -1
// true
// Infinity -Infinity
```

## 6 格式化返回字符串

toFixed(digits) 通常按小数点后 digits 位输出，digits 范围为 0–100；绝对值达到 10 ** 21 时采用数值字符串转换的形式。toPrecision(precision) 按有效数字位数输出，precision 范围为 1–100；toExponential() 使用指数形式。

这些方法返回字符串，用于显示，不改变底层 Number，也不能修复此前的浮点误差。toFixed() 的舍入针对实际存储的二进制数值，不能仅按源码十进制外观推断。带区域格式的显示在日期与国际化章节展开。

```javascript
const distance = 12.5;
console.log(distance.toFixed(2), distance.toPrecision(4), distance.toExponential(2));
console.log(typeof distance.toFixed(2), distance);
console.log((2.55).toFixed(1), (1e21).toFixed(2));
// 输出依次为：
// 12.50 12.50 1.25e+1
// string 12.5
// 2.5 1e+21
```

## 7 BigInt 的精度与转换边界

BigInt 表示任意精度整数，但仍受实现内存等资源限制。用字符串或 n 字面量构造大整数，避免先经过不安全的 Number。BigInt() 可显式接收整数 Number；已有的精度损失无法恢复，非整数 Number 会抛出 RangeError。

BigInt 的除法向零截断，除以 0n 抛出 RangeError，没有 Infinity 或 NaN。算术与位运算两边必须同为 BigInt，不能混入 Number；一元加不支持 BigInt。关系比较和宽松相等有各自的跨数值类型规则，不属于混合算术。转回 Number 可能失去精度。

```javascript
const exact = BigInt("9007199254740993");
console.log(exact, exact + 2n, 7n / 2n, -7n / 2n);
console.log(BigInt(9007199254740993), Number(exact));
console.log(2n < 3, 2n == 2, 2n === 2);
console.log(BigInt(12), Number(12n));
// 输出依次为：
// 9007199254740993n 9007199254740995n 3n -3n
// 9007199254740992n 9007199254740992
// true true false
// 12n 12
```

## 8 随机数的用途边界

Math.random() 返回大于等于 0、小于 1 的 Number，算法由实现决定，标准接口不提供设置随机种子的参数。可用于普通模拟和非安全的随机展示，不能据此生成密码或安全令牌；安全随机源属于宿主能力。

下面 count 是选项数量 6，sample 是给定的样本值，用 Math.floor(sample * count) 观察下标映射。真实 Math.random() 只核对范围，不把某一次输出当作固定答案；也不以少量样本证明分布质量。

```javascript
const countOptions = 6;
const sample = 0.42; // 固定输入只演示映射，不冒充随机测量
console.log(Math.floor(sample * countOptions));
const randomValue = Math.random();
console.log(randomValue >= 0 && randomValue < 1);
// 输出依次为：
// 2
// true
```

## 9 独立观察错误边界

以下文件分别启动新进程；预期退出码为 1。先根据代码判断错误原因，再运行相应命令，核对错误名称及对应位置。错误消息全文由宿主决定。

BigInt 与 Number 不能直接相加；先决定所需的数值类型。

```javascript
console.log(1n + 1);
// 预期错误：TypeError；Cannot mix BigInt and other types
```

Step 1：独立运行 scripts/04-numbers-and-math/mixed-numeric-types.mjs。

```bash
node scripts/04-numbers-and-math/mixed-numeric-types.mjs
```

非整数 Number 不能直接转换为 BigInt。

```javascript
console.log(BigInt(1.5));
// 预期错误：RangeError；cannot be converted to a BigInt because it is not an integer
```

Step 2：独立运行 scripts/04-numbers-and-math/bigint-fraction.mjs。

```bash
node scripts/04-numbers-and-math/bigint-fraction.mjs
```

BigInt 除以零不会产生 Infinity。

```javascript
console.log(1n / 0n);
// 预期错误：RangeError；Division by zero
```

Step 3：独立运行 scripts/04-numbers-and-math/bigint-zero-division.mjs。

```bash
node scripts/04-numbers-and-math/bigint-zero-division.mjs
```

## 本章小结

- Number 的“整数”不等于安全整数，显示出的十进制也不等于内部精确十进制。
- 分清转换、前缀解析、业务校验与格式化四个用途。
- BigInt 适合精确整数；Math.random() 只提供普通随机用途所需的语言接口。

## 练习

1. 分别解析 "15.8kg" 和 "15.8"；核对 Number() 只接受后者，parseFloat() 都得到 15.8，并说明何时必须拒绝前者。
2. 对 -2.5、2.5 比较四种取整方式；标准是 floor 不大于输入，ceil 不小于输入，trunc 朝零，round 等距朝正无穷。
3. 使用 BigInt 求 9007199254740993 与 7 的整数和；核对得到 9007199254741000n，并确认创建输入时没有经过 Number。
4. 用给定样本 0、0.5、0.999 将 6 个选项映射到下标；核对 0、3、5，不声称这些样本测试了随机源。

## 参考与引用来源

- TC39（tc39.es）：[§6.1.6 Number 与 BigInt](https://tc39.es/ecma262/2025/multipage/ecmascript-data-types-and-values.html#sec-numeric-types)、[§12.9.3 数值字面量](https://tc39.es/ecma262/2025/multipage/ecmascript-language-lexical-grammar.html#sec-literals-numeric-literals)、[§21.1 Number](https://tc39.es/ecma262/2025/multipage/numbers-and-dates.html#sec-number-objects)、[§21.2 BigInt](https://tc39.es/ecma262/2025/multipage/numbers-and-dates.html#sec-bigint-objects)、[§21.3 Math](https://tc39.es/ecma262/2025/multipage/numbers-and-dates.html#sec-math-object)、[§19.2.2–19.2.5 有限值检查与解析](https://tc39.es/ecma262/2025/multipage/global-object.html#sec-parseint-string-radix)：ECMAScript 2025 的精度、解析、舍入、格式化与随机数规则。
- MDN：[Math.random() 用途对照](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Math/random)：非密码学随机数、种子接口与区间映射的使用边界。
- Node.js：[24.11.0 crypto.randomBytes](https://nodejs.org/download/release/v24.11.0/docs/api/crypto.html#cryptorandombytessize-callback)：密码学安全伪随机源属于宿主 API，与 Math.random() 的语言接口区分。